# Task 3: Occasion and Gender Classification


## 1. Introduction

This notebook addresses **Task 3: Occasion and Gender Classification** for Assignment 2 by predicting the catalogue gender category and intended occasion of a fashion product from its RGB image. The task is formulated as **two separate multi-class classification problems: one model predicts gender and another predicts occasion (the metadata field `usage`)**.

Three neural-network approaches are developed and trained from scratch using TensorFlow/Keras:

- **Shallow MLP (Baseline):** A fully connected network with one 256-unit hidden layer operating on flattened image pixels.
- **Deeper MLP:** A fully connected network with hidden layers of 256, 128, and 64 units, using dropout to reduce overfitting.
- **Convolutional Neural Network (CNN):** A spatial model that learns local image patterns through convolutional layers. The CNN family includes three declared architecture/scheduling configurations.

Performance is assessed using:

- **Accuracy:** The proportion of correctly classified images.
- **Macro-F1:** The primary selection metric, giving equal weight to the F1 scores of classes present in the evaluation partition.
- **Per-class classification reports and confusion matrices:** Evidence of which labels are recognized reliably and which are confused.
- **Learning curves:** Training and validation loss, accuracy, and macro-F1 used to examine convergence and overfitting.
- **Calibration metrics:** Confidence reliability of the selected model, assessed after selection using separate validation groups.

Gender labels describe the catalogue category rather than the identity of a person in an image. Occasion labels may overlap visually, and frequent categories can dominate accuracy. Each target is trained, compared, and evaluated separately.

The workflow covers metadata inspection, preprocessing, model development, comparative evaluation, selection, and export for prediction. All candidates use the same frozen group-isolated data partitions. The internal test has prior development exposure, which remains a limitation after retraining. Numerical findings and the final model judgment must be completed from the new executed results; no winner is assumed in advance.


## 2. Library Imports & Setup

Use the Fashion Keras kernel. The seed controls initialization and augmentation. Float32 is used throughout; GPU availability is reported before models are created.


In [ ]:
# Locate the project, import the training libraries, select the accelerator, and fix random seeds.
from pathlib import Path
import sys, json
from concurrent.futures import ThreadPoolExecutor

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageEnhance
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    classification_report,
    log_loss,
    ConfusionMatrixDisplay,
)
from sklearn.model_selection import GroupShuffleSplit
from scipy.optimize import minimize_scalar
import tensorflow as tf
from tensorflow import keras
from scripts.preprocessing import (
    task_frame,
    IMAGE_SIZE,
    NORMALISATION_PATH,
    SEED,
    select_tensorflow_device,
)

DEVICE = select_tensorflow_device()
keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()
OUTPUT, RESULTS, FIGURES = (ROOT / 'models', ROOT / 'results', ROOT / 'figures')
for directory in (OUTPUT, RESULTS, FIGURES):
    directory.mkdir(parents=True, exist_ok=True)
MAX_EPOCHS, BATCH_SIZE = (30, 64)

def stem_for(target):
    return 'article_type' if target == 'articleType' else target

get_ipython().run_line_magic('matplotlib', 'inline')

# False for a fresh Run All. Enable only when deliberately resuming matching saved experiments.
RESUME_SAVED_RESULTS = False

## 3. Load Metadata

Load valid labelled images and their frozen split assignments. Inspect paths, target labels and missing values before preprocessing.


In [ ]:
# Load audited metadata and show how many labelled images are available in each frozen split.
metadata_by_target = {'gender': task_frame('gender'), 'usage': task_frame('usage')}
frame = metadata_by_target['gender']
print('gender', frame.shape)
display(frame.head())
frame = metadata_by_target['usage']
print('usage', frame.shape)
display(frame.head())

## 4. Data Preprocessing

Prepare the same images and label encoding for every candidate. Preserve group isolation and fit normalization on training data only.


### 4.1. Class Distribution & Balancing Strategy

Preserve the original sample distribution. Initial candidates use unweighted cross-entropy; additional CNN experiments compare this with capped square-root inverse-frequency weights computed only from training counts. No examples are duplicated, and validation metrics remain unweighted.


In [ ]:
# Measure training-set class imbalance before choosing architectures and evaluation metrics.
train_rows = metadata_by_target['gender'].loc[metadata_by_target['gender']['split'].eq('train')]
counts = train_rows['gender'].value_counts()
counts.head(25).sort_values().plot.barh(figsize=(8, 6), title=f"{'gender'}: training support")
plt.tight_layout()
plt.show()
train_rows = metadata_by_target['usage'].loc[metadata_by_target['usage']['split'].eq('train')]
counts = train_rows['usage'].value_counts()
counts.head(25).sort_values().plot.barh(figsize=(8, 6), title=f"{'usage'}: training support")
plt.tight_layout()
plt.show()

### 4.2. Training, Validation & Test Partitions

Reuse the frozen product-group split. Within validation, use separate groups for selection, temperature fitting and policy checking. Do not use the internal test to select candidates.


In [ ]:
# Divide validation groups into model-selection and calibration views without splitting related products.
def validation_views(frame):
    selection_ids, rest_ids = next(
        GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=SEED).split(
            frame, groups=frame.group_key
        )
    )
    selection, rest = frame.iloc[selection_ids], frame.iloc[rest_ids]
    cal_ids, policy_ids = next(
        GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=SEED + 1).split(
            rest, groups=rest.group_key
        )
    )
    return selection, rest.iloc[cal_ids], rest.iloc[policy_ids]

In [ ]:
# Create the split views, fixed label order, and reusable batch loaders for each target.
frames, labels_by_target = ({}, {})
selection, calibration, policy = validation_views(
    metadata_by_target['gender'].loc[metadata_by_target['gender']['split'].eq('validation')]
)
frames['gender'] = dict(
    train=metadata_by_target['gender'].loc[metadata_by_target['gender']['split'].eq('train')],
    selection=selection,
    calibration=calibration,
    policy=policy,
)
groups = [set(frame.group_key) for frame in frames['gender'].values()]
assert all((a.isdisjoint(b) for i, a in enumerate(groups) for b in groups[i + 1 :]))
labels_by_target['gender'] = sorted(frames['gender']['train']['gender'].unique())
display(pd.Series({name: len(frame) for name, frame in frames['gender'].items()}, name='gender'))
selection, calibration, policy = validation_views(
    metadata_by_target['usage'].loc[metadata_by_target['usage']['split'].eq('validation')]
)
frames['usage'] = dict(
    train=metadata_by_target['usage'].loc[metadata_by_target['usage']['split'].eq('train')],
    selection=selection,
    calibration=calibration,
    policy=policy,
)
groups = [set(frame.group_key) for frame in frames['usage'].values()]
assert all((a.isdisjoint(b) for i, a in enumerate(groups) for b in groups[i + 1 :]))
labels_by_target['usage'] = sorted(frames['usage']['train']['usage'].unique())
display(pd.Series({name: len(frame) for name, frame in frames['usage'].items()}, name='usage'))

### 4.3. Image Preprocessing

Resize RGB to 96 x 128 pixels (width x height), then normalize using training-only channel statistics. The batch class stores decoded uint8 images and converts one batch at a time to float32. Horizontal flips are applied inside each model only during training.


In [ ]:
# Cache resized images in RAM and serve normalized batches to Keras during training and evaluation.
class CachedBatches(keras.utils.PyDataset):
    """Decode RGB once to uint8 RAM, then normalize each NHWC batch."""

    def __init__(self, frame, target, labels, normalisation, training=False, batch_size=64):
        super().__init__(workers=0, max_queue_size=2)
        self.dataset = frame
        self.training, self.batch_size = training, batch_size

        def read(path):
            with Image.open(path) as image:
                return np.asarray(
                    image.convert("RGB").resize(IMAGE_SIZE, Image.Resampling.BILINEAR)
                ).copy()

        # Decode and resize each source image once; later epochs reuse this cache.
        with ThreadPoolExecutor(max_workers=4) as pool:
            self.images = np.stack(list(pool.map(read, frame.image_path)))

        # Convert string labels to the fixed integer order expected by the output layer.
        self.targets = np.asarray([labels.index(label) for label in frame[target]], dtype=np.int32)
        self.mean = np.asarray(normalisation["mean"], dtype=np.float32)
        self.std = np.asarray(normalisation["std"], dtype=np.float32)
        self.reset()

    def reset(self):
        self.rng = np.random.default_rng(SEED)
        self.indices = np.arange(len(self.dataset))
        if self.training:
            self.rng.shuffle(self.indices)

    def __len__(self):
        return (len(self.dataset) + self.batch_size - 1) // self.batch_size

    # Normalize only the requested batch to avoid storing a second float32 image copy.
    def __getitem__(self, index):
        ids = self.indices[index * self.batch_size : (index + 1) * self.batch_size]
        images = self.images[ids].astype(np.float32) / 255.0
        return (images - self.mean) / self.std, self.targets[ids]

    def on_epoch_end(self):
        if self.training:
            self.rng.shuffle(self.indices)

### 4.4. Feature Batches & Label Encoding

Map sorted label names to integer indices, preserving the same order for every model. Construct shuffled training batches and fixed-order selection batches.


In [ ]:
# Load normalization statistics computed from training images only, then construct all data loaders.
normalisation = json.loads(NORMALISATION_PATH.read_text())
loaders = {}
labels = labels_by_target['gender']
loaders['gender'] = {
    name: CachedBatches(
        frames['gender'][name],
        'gender',
        labels,
        normalisation,
        training=name == 'train',
        batch_size=BATCH_SIZE,
    )
    for name in ['train', 'selection']
}
labels = labels_by_target['usage']
loaders['usage'] = {
    name: CachedBatches(
        frames['usage'][name],
        'usage',
        labels,
        normalisation,
        training=name == 'train',
        batch_size=BATCH_SIZE,
    )
    for name in ['train', 'selection']
}

## 5. Model Development & Evaluation

Train and evaluate a shallow MLP baseline, a deeper MLP and a CNN using the same image preprocessing and frozen partitions. Architecture construction, fitting, class-level evaluation, calibration and model export are implemented in this notebook.


### Evaluation Metric: Macro-F1

Accumulate the full-epoch confusion matrix and average F1 over classes with positive ground-truth support. This metric selects the restored epoch and winning model. It does not average per-batch F1 scores.

The displayed per-class classification report also includes every output label for coverage auditing. Its standard `macro avg` row therefore includes zero-support output labels and can differ from the supported-class macro-F1 used for selection. Read the explicitly reported selection metric for model ranking, and inspect support before making claims about all classes. Selection support is 97/124 article types, 4/4 seasons, 5/5 gender categories and 8/9 occasion labels for this frozen run.


In [ ]:
# Track macro-F1 across a complete epoch with a running confusion matrix.
@keras.utils.register_keras_serializable(package="Fashion")
class SupportedMacroF1(keras.metrics.Metric):
    """Macro-F1 over classes present in the full epoch's ground truth."""

    def __init__(self, num_classes, name="macro_f1", **kwargs):
        super().__init__(name=name, **kwargs)
        self.num_classes = num_classes
        self.matrix = self.add_weight(
            name="matrix", shape=(num_classes, num_classes), initializer="zeros"
        )

    # Accumulate one confusion matrix across all batches in the epoch.
    def update_state(self, y_true, y_pred, sample_weight=None):
        truth = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        predictions = tf.argmax(y_pred, axis=-1, output_type=tf.int32)
        weights = (
            None if sample_weight is None else tf.cast(tf.reshape(sample_weight, [-1]), self.dtype)
        )
        self.matrix.assign_add(
            tf.math.confusion_matrix(
                truth, predictions, self.num_classes, weights=weights, dtype=self.dtype
            )
        )

    # Average F1 only across classes that are present in the ground truth.
    def result(self):
        support = tf.reduce_sum(self.matrix, axis=1)
        predicted = tf.reduce_sum(self.matrix, axis=0)
        f1 = tf.math.divide_no_nan(2 * tf.linalg.diag_part(self.matrix), support + predicted)
        mask = tf.cast(support > 0, self.dtype)
        return tf.math.divide_no_nan(tf.reduce_sum(f1 * mask), tf.reduce_sum(mask))

    def reset_state(self):
        self.matrix.assign(tf.zeros_like(self.matrix))

    def get_config(self):
        return {**super().get_config(), "num_classes": self.num_classes}

### Saved-Model Metadata

An identity layer stores label order, preprocessing and confidence policy inside each exported Keras model.


In [ ]:
# Attach labels and preprocessing settings to the exported Keras model as a pass-through layer.
@keras.utils.register_keras_serializable(package="Fashion")
class ModelMetadata(keras.layers.Layer):
    """Store label order, preprocessing and calibration inside the .keras file."""

    def __init__(self, metadata=None, **kwargs):
        super().__init__(**kwargs)
        self.metadata = dict(metadata or {})

    def call(self, inputs):
        return inputs

    def get_config(self):
        return {**super().get_config(), "metadata": dict(self.metadata)}

In [ ]:
# Keep trained models, learning histories, and selection scores organized by target and method.
models = {'gender': {}, 'usage': {}}
histories = {'gender': {}, 'usage': {}}
selection_scores = {}

### Configuration rationale and evaluation protocol

Each target compares three neural model families. Dense width and depth control model capacity; convolutional blocks preserve spatial relationships. Configuration choices follow validation macro-F1, then accuracy, then parameter count. The selected architectures and training settings are specified below; they are empirical choices, not theoretically optimal designs.

The 96 ? 128 RGB input and training-only normalization are shared across candidates. Most source images are only 60 ? 80, so enlarging them further would not recover additional image detail. Output width is determined by the target's training label vocabulary, rather than tuned independently.

Where required, CNN training has an initial four-block stage followed by a lower-learning-rate continuation. Both stages are implemented here. Class weights use training counts only. The initial stage is part of the training recipe, not a fourth model family.

The recorded tables summarize completed runs of these configurations. This reorganized notebook has not yet been executed end-to-end; rerunning the cells regenerates its metrics, figures and exports. Single-seed results may vary. Only the supplied dataset, shared preprocessing module and frozen Task 0 preprocessing artifacts are required; all model training code is included here.


### General design justification

**Input and output.** A fixed 96 ? 128 RGB input standardizes batching and inference while keeping computation manageable. Its portrait aspect ratio matches the dominant 60 ? 80 source format. Resizing cannot recover missing detail, so larger inputs are not assumed to improve recognition. Output units match the target's label vocabulary; they are not a freely tuned capacity setting.

**Dense baselines.** Flattening provides a straightforward pixel-based baseline. A shallow MLP tests whether one nonlinear hidden layer is sufficient; a deeper MLP tests additional nonlinear transformations. Dense layers do not explicitly preserve local spatial relationships, and flattening creates large parameter counts. ReLU introduces nonlinearity. Hidden widths and depths are selected using validation evidence, not the number of classes alone.

**CNN architecture.** Small 3 ? 3 convolutions share weights across positions and can learn local appearance patterns. Successive blocks allow information from larger image regions to be combined. Max pooling reduces spatial dimensions and computational cost, but can discard fine detail. The final 2 ? 2 average-pooled grid limits the dense head's size while retaining a coarse spatial layout. Batch normalization stabilizes intermediate activation scales. These design properties motivate CNN use; they do not prove that a particular learned filter detects a specific garment feature.

**Regularization and optimization.** Dropout 0.2 discourages reliance on individual hidden activations, while training-only horizontal flips expose models to mirrored product views. These are attempts to improve generalization, not guarantees. Adam adapts parameter updates; reducing its learning rate on a validation plateau allows smaller updates later in CNN training. Early stopping restores the best selection macro-F1 epoch and limits training after improvement stalls. The fixed seed supports repeatability, and batch size 64 is a practical training setting rather than a demonstrated optimum.

**Model selection.** Group-isolated partitions reduce related-product leakage, training-only normalization avoids fitting preprocessing to evaluation data, and macro-F1 gives each supported class equal weight. Accuracy, class-level errors and parameter counts provide complementary evidence. Different labels may favour different capacities; the measured validation comparison justifies the exact choices, while the general principles above explain their intended roles.


In [ ]:
SELECTED_CONFIGS = {
    'gender': {
        'shallow_mlp': {
            'method': 'shallow_mlp_lower_lr',
            'widths': [256],
            'learning_rate': 0.0001,
            'epochs': 20,
            'patience': 4,
        },
        'deeper_mlp': {
            'method': 'deeper_mlp_two_layers',
            'widths': [256, 128],
            'learning_rate': 0.0003,
            'epochs': 20,
            'patience': 4,
        },
        'cnn': {
            'method': 'cnn_four_blocks_scheduled',
            'widths': [256],
            'learning_rate': 0.001,
            'epochs': 30,
            'patience': 7,
        },
    },
    'usage': {
        'shallow_mlp': {
            'method': 'shallow_mlp_lower_lr',
            'widths': [256],
            'learning_rate': 0.0001,
            'epochs': 20,
            'patience': 4,
        },
        'deeper_mlp': {
            'method': 'deeper_mlp',
            'widths': [256, 128, 64],
            'learning_rate': 0.001,
            'epochs': 30,
            'patience': 7,
        },
        'cnn': {
            'method': 'cnn_continue_weighted',
            'widths': [256],
            'learning_rate': 3e-05,
            'epochs': 30,
            'patience': 7,
        },
    },
}

### 5.1. Shallow MLP (Baseline)

Flattened RGB input is passed through the selected dense layers, with ReLU and dropout 0.2 after each hidden layer.


#### 5.1.1. Gender: Model Architecture & Training Configuration

Configuration `shallow_mlp_lower_lr`; hidden dense units [256]. Adam 0.0001, maximum 20 epochs, early-stopping patience 4.


In [ ]:
# Build the shallow MLP from the selected target-specific configuration.
config = SELECTED_CONFIGS['gender']['shallow_mlp']
method = config['method']
keras.utils.set_random_seed(SEED)

# Start with a fixed image input and training-only horizontal augmentation.
layers = [
    keras.Input(shape=(IMAGE_SIZE[1], IMAGE_SIZE[0], 3)),
    keras.layers.RandomFlip('horizontal'),
]
layers.append(keras.layers.Flatten())

# Add the configured hidden classifier layers and dropout regularization.
for width in config['widths']:
    layers.extend([keras.layers.Dense(width, activation='relu'), keras.layers.Dropout(0.2)])
layers.extend([keras.layers.Dense(len(labels_by_target['gender'])), ModelMetadata(name='metadata')])
models['gender'][method] = keras.Sequential(layers, name=method)
models['gender'][method].summary()

#### 5.1.2. Gender: Compile & Train Model

The selection view controls epoch selection. Test images are excluded from training.


In [ ]:
# Compile and train the model while restoring the epoch with the best selection macro-F1.
model = models['gender'][method]

# Configure optimization and report both accuracy and imbalance-aware macro-F1.
model.compile(
    optimizer=keras.optimizers.Adam(0.0001),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[
        keras.metrics.SparseCategoricalAccuracy(name='accuracy'),
        SupportedMacroF1(len(labels_by_target['gender'])),
    ],
)
loaders['gender']['train'].reset()

# Stop when selection macro-F1 no longer improves and restore its best weights.
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_macro_f1', mode='max', patience=4, restore_best_weights=True
    )
]

# Keep the selection partition read-only: it chooses epochs but never updates weights.
fitted = model.fit(
    loaders['gender']['train'],
    validation_data=loaders['gender']['selection'],
    epochs=20,
    callbacks=callbacks,
    shuffle=False,
    verbose=2,
)
histories['gender'][method] = pd.DataFrame(fitted.history)

#### 5.1.3. Gender: Evaluation and Performance Metrics

Calculate validation accuracy and supported-class macro-F1, display the per-class classification report, and plot training and validation loss, accuracy and macro-F1. These measurements use the restored best epoch.


In [ ]:
# Evaluate the restored best weights, show class metrics, and plot the learning curves.
loader = loaders['gender']['selection']

# Convert logits to probabilities and preserve the loader's row order.
probabilities = np.vstack(
    [
        tf.nn.softmax(models['gender'][method](loader[i][0], training=False)).numpy()
        for i in range(len(loader))
    ]
)
predictions = probabilities.argmax(1)
display(
    pd.Series(
        dict(
            selection_accuracy=accuracy_score(loader.targets, predictions),
            selection_macro_f1=f1_score(
                loader.targets,
                predictions,
                labels=np.unique(loader.targets),
                average='macro',
                zero_division=0,
            ),
        )
    )
)

# Inspect every label because aggregate accuracy can hide minority-class failures.
per_class = pd.DataFrame(
    classification_report(
        loader.targets,
        predictions,
        labels=np.arange(len(labels_by_target['gender'])),
        target_names=labels_by_target['gender'],
        output_dict=True,
        zero_division=0,
    )
).T
display(per_class)
per_class.to_csv(RESULTS / f"{stem_for('gender')}_{method}_selection_per_class.csv")

# Plot train and selection curves to diagnose convergence and overfitting.
history = histories['gender'][method]
history.to_csv(RESULTS / f"{stem_for('gender')}_{method}_history.csv", index=False)
fig, axes = plt.subplots(1, 3, figsize=(14, 3))
for ax, metric in zip(axes, ['loss', 'accuracy', 'macro_f1']):
    ax.plot(np.arange(1, len(history) + 1), history[metric], label='Train')
    ax.plot(np.arange(1, len(history) + 1), history['val_' + metric], label='Selection')
    ax.set(xlabel='Epoch', ylabel=metric, title=method)
    ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / f"{stem_for('gender')}_{method}_learning_curves.png", dpi=160)
plt.show()

# Store one comparable row for the final cross-family decision.
selection_scores.setdefault('gender', {})[method] = dict(
    family='shallow_mlp',
    validation_accuracy=accuracy_score(loader.targets, predictions),
    validation_macro_f1=f1_score(
        loader.targets,
        predictions,
        labels=np.unique(loader.targets),
        average='macro',
        zero_division=0,
    ),
    complexity_parameters=models['gender'][method].count_params(),
    epochs_run=len(history),
)

#### 5.1.4. Gender: Evaluation Analysis

**Learning behaviour.** The shallow MLP reached its best gender macro-F1 at epoch 8 of 12. Training accuracy was **85.00%** and selection accuracy was **84.69%**, while training and selection losses were **0.4318** and **0.4602**. The curves are closely aligned, indicating that dropout, augmentation and early stopping prevented marked overfitting. The **0.7118 macro-F1** is lower than accuracy because the five gender classes are highly unequal in size.

**Class-level behaviour.** Men and Women, with 1,612 and 1,001 selection images, achieved F1 scores of **0.892** and **0.839**. Boys also reached **0.748 F1**, while Girls achieved **0.606**. Unisex was the clearest weakness: precision was **0.690**, but recall was only **0.360**, giving **0.473 F1**. Thus, when the model predicted Unisex it was often correct, but it missed nearly two-thirds of the true Unisex examples.

**Interpretation.** The shallow ANN forms a strong accuracy baseline for gender because the two dominant classes are well learned. Its lower macro-F1 reveals that overall accuracy overstates reliability for Unisex and Girls. This imbalance-sensitive result is why macro-F1, together with the per-class metrics, is needed for model selection.


#### 5.1.5. Occasion: Model Architecture & Training Configuration

Configuration `shallow_mlp_lower_lr`; hidden dense units [256]. Adam 0.0001, maximum 20 epochs, early-stopping patience 4.


In [ ]:
# Build the shallow MLP from the selected target-specific configuration.
config = SELECTED_CONFIGS['usage']['shallow_mlp']
method = config['method']
keras.utils.set_random_seed(SEED)

# Start with a fixed image input and training-only horizontal augmentation.
layers = [
    keras.Input(shape=(IMAGE_SIZE[1], IMAGE_SIZE[0], 3)),
    keras.layers.RandomFlip('horizontal'),
]
layers.append(keras.layers.Flatten())

# Add the configured hidden classifier layers and dropout regularization.
for width in config['widths']:
    layers.extend([keras.layers.Dense(width, activation='relu'), keras.layers.Dropout(0.2)])
layers.extend([keras.layers.Dense(len(labels_by_target['usage'])), ModelMetadata(name='metadata')])
models['usage'][method] = keras.Sequential(layers, name=method)
models['usage'][method].summary()

#### 5.1.6. Occasion: Compile & Train Model

The selection view controls epoch selection. Test images are excluded from training.


In [ ]:
# Compile and train the model while restoring the epoch with the best selection macro-F1.
model = models['usage'][method]

# Configure optimization and report both accuracy and imbalance-aware macro-F1.
model.compile(
    optimizer=keras.optimizers.Adam(0.0001),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[
        keras.metrics.SparseCategoricalAccuracy(name='accuracy'),
        SupportedMacroF1(len(labels_by_target['usage'])),
    ],
)
loaders['usage']['train'].reset()

# Stop when selection macro-F1 no longer improves and restore its best weights.
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_macro_f1', mode='max', patience=4, restore_best_weights=True
    )
]

# Keep the selection partition read-only: it chooses epochs but never updates weights.
fitted = model.fit(
    loaders['usage']['train'],
    validation_data=loaders['usage']['selection'],
    epochs=20,
    callbacks=callbacks,
    shuffle=False,
    verbose=2,
)
histories['usage'][method] = pd.DataFrame(fitted.history)

#### 5.1.7. Occasion: Evaluation and Performance Metrics

Calculate validation accuracy and supported-class macro-F1, display the per-class classification report, and plot training and validation loss, accuracy and macro-F1. These measurements use the restored best epoch.


In [ ]:
# Evaluate the restored best weights, show class metrics, and plot the learning curves.
loader = loaders['usage']['selection']

# Convert logits to probabilities and preserve the loader's row order.
probabilities = np.vstack(
    [
        tf.nn.softmax(models['usage'][method](loader[i][0], training=False)).numpy()
        for i in range(len(loader))
    ]
)
predictions = probabilities.argmax(1)
display(
    pd.Series(
        dict(
            selection_accuracy=accuracy_score(loader.targets, predictions),
            selection_macro_f1=f1_score(
                loader.targets,
                predictions,
                labels=np.unique(loader.targets),
                average='macro',
                zero_division=0,
            ),
        )
    )
)

# Inspect every label because aggregate accuracy can hide minority-class failures.
per_class = pd.DataFrame(
    classification_report(
        loader.targets,
        predictions,
        labels=np.arange(len(labels_by_target['usage'])),
        target_names=labels_by_target['usage'],
        output_dict=True,
        zero_division=0,
    )
).T
display(per_class)
per_class.to_csv(RESULTS / f"{stem_for('usage')}_{method}_selection_per_class.csv")

# Plot train and selection curves to diagnose convergence and overfitting.
history = histories['usage'][method]
history.to_csv(RESULTS / f"{stem_for('usage')}_{method}_history.csv", index=False)
fig, axes = plt.subplots(1, 3, figsize=(14, 3))
for ax, metric in zip(axes, ['loss', 'accuracy', 'macro_f1']):
    ax.plot(np.arange(1, len(history) + 1), history[metric], label='Train')
    ax.plot(np.arange(1, len(history) + 1), history['val_' + metric], label='Selection')
    ax.set(xlabel='Epoch', ylabel=metric, title=method)
    ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / f"{stem_for('usage')}_{method}_learning_curves.png", dpi=160)
plt.show()

# Store one comparable row for the final cross-family decision.
selection_scores.setdefault('usage', {})[method] = dict(
    family='shallow_mlp',
    validation_accuracy=accuracy_score(loader.targets, predictions),
    validation_macro_f1=f1_score(
        loader.targets,
        predictions,
        labels=np.unique(loader.targets),
        average='macro',
        zero_division=0,
    ),
    complexity_parameters=models['usage'][method].count_params(),
    epochs_run=len(history),
)

#### 5.1.8. Occasion: Evaluation Analysis

**Learning behaviour.** For occasion, the shallow MLP reached its best macro-F1 at epoch 10 of 14. Training and selection accuracy were **86.81%** and **87.38%**, with losses of **0.4097** and **0.4874**. The accuracy curves show little overfitting, but the model's **0.4522 macro-F1** is far below its accuracy. This discrepancy is caused by the extreme dominance of Casual and cannot be understood from accuracy alone.

**Class-level behaviour.** Casual accounted for 2,237 selection images and achieved **0.922 F1**, strongly influencing overall accuracy. Ethnic and Formal reached **0.758** and **0.751 F1**, whereas Sports recall was only **0.528** and its F1 was **0.615**. The rare NA/missing-usage category recovered two of five examples. Party, Smart Casual and Travel, with only one, four and two examples respectively, all received zero recall.

**Interpretation.** The shallow model handles the common catalogue occasions reasonably well but does not provide dependable coverage of extremely rare labels. Its result is a useful baseline; however, the large accuracy–macro-F1 difference demonstrates that deployment quality depends on more than correct Casual predictions. Additional dense depth or a spatial feature extractor must be judged primarily by whether it improves the minority classes.


### 5.2. Deeper MLP

Flattened RGB input is passed through the selected dense layers, with ReLU and dropout 0.2 after each hidden layer.


#### 5.2.1. Gender: Model Architecture & Training Configuration

Configuration `deeper_mlp_two_layers`; hidden dense units [256, 128]. Adam 0.0003, maximum 20 epochs, early-stopping patience 4.


In [ ]:
# Build the deeper MLP from the selected target-specific configuration.
config = SELECTED_CONFIGS['gender']['deeper_mlp']
method = config['method']
keras.utils.set_random_seed(SEED)

# Start with a fixed image input and training-only horizontal augmentation.
layers = [
    keras.Input(shape=(IMAGE_SIZE[1], IMAGE_SIZE[0], 3)),
    keras.layers.RandomFlip('horizontal'),
]
layers.append(keras.layers.Flatten())

# Add the configured hidden classifier layers and dropout regularization.
for width in config['widths']:
    layers.extend([keras.layers.Dense(width, activation='relu'), keras.layers.Dropout(0.2)])
layers.extend([keras.layers.Dense(len(labels_by_target['gender'])), ModelMetadata(name='metadata')])
models['gender'][method] = keras.Sequential(layers, name=method)
models['gender'][method].summary()

#### 5.2.2. Gender: Compile & Train Model

The selection view controls epoch selection. Test images are excluded from training.


In [ ]:
# Compile and train the model while restoring the epoch with the best selection macro-F1.
model = models['gender'][method]

# Configure optimization and report both accuracy and imbalance-aware macro-F1.
model.compile(
    optimizer=keras.optimizers.Adam(0.0003),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[
        keras.metrics.SparseCategoricalAccuracy(name='accuracy'),
        SupportedMacroF1(len(labels_by_target['gender'])),
    ],
)
loaders['gender']['train'].reset()

# Stop when selection macro-F1 no longer improves and restore its best weights.
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_macro_f1', mode='max', patience=4, restore_best_weights=True
    )
]

# Keep the selection partition read-only: it chooses epochs but never updates weights.
fitted = model.fit(
    loaders['gender']['train'],
    validation_data=loaders['gender']['selection'],
    epochs=20,
    callbacks=callbacks,
    shuffle=False,
    verbose=2,
)
histories['gender'][method] = pd.DataFrame(fitted.history)

#### 5.2.3. Gender: Evaluation and Performance Metrics

Calculate validation accuracy and supported-class macro-F1, display the per-class classification report, and plot training and validation loss, accuracy and macro-F1. These measurements use the restored best epoch.


In [ ]:
# Evaluate the restored best weights, show class metrics, and plot the learning curves.
loader = loaders['gender']['selection']

# Convert logits to probabilities and preserve the loader's row order.
probabilities = np.vstack(
    [
        tf.nn.softmax(models['gender'][method](loader[i][0], training=False)).numpy()
        for i in range(len(loader))
    ]
)
predictions = probabilities.argmax(1)
display(
    pd.Series(
        dict(
            selection_accuracy=accuracy_score(loader.targets, predictions),
            selection_macro_f1=f1_score(
                loader.targets,
                predictions,
                labels=np.unique(loader.targets),
                average='macro',
                zero_division=0,
            ),
        )
    )
)

# Inspect every label because aggregate accuracy can hide minority-class failures.
per_class = pd.DataFrame(
    classification_report(
        loader.targets,
        predictions,
        labels=np.arange(len(labels_by_target['gender'])),
        target_names=labels_by_target['gender'],
        output_dict=True,
        zero_division=0,
    )
).T
display(per_class)
per_class.to_csv(RESULTS / f"{stem_for('gender')}_{method}_selection_per_class.csv")

# Plot train and selection curves to diagnose convergence and overfitting.
history = histories['gender'][method]
history.to_csv(RESULTS / f"{stem_for('gender')}_{method}_history.csv", index=False)
fig, axes = plt.subplots(1, 3, figsize=(14, 3))
for ax, metric in zip(axes, ['loss', 'accuracy', 'macro_f1']):
    ax.plot(np.arange(1, len(history) + 1), history[metric], label='Train')
    ax.plot(np.arange(1, len(history) + 1), history['val_' + metric], label='Selection')
    ax.set(xlabel='Epoch', ylabel=metric, title=method)
    ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / f"{stem_for('gender')}_{method}_learning_curves.png", dpi=160)
plt.show()

# Store one comparable row for the final cross-family decision.
selection_scores.setdefault('gender', {})[method] = dict(
    family='deeper_mlp',
    validation_accuracy=accuracy_score(loader.targets, predictions),
    validation_macro_f1=f1_score(
        loader.targets,
        predictions,
        labels=np.unique(loader.targets),
        average='macro',
        zero_division=0,
    ),
    complexity_parameters=models['gender'][method].count_params(),
    epochs_run=len(history),
)

#### 5.2.4. Gender: Evaluation Analysis

**Learning behaviour.** The deeper MLP obtained its best gender macro-F1 at epoch 18 of 20. Training accuracy was **84.07%**, selection accuracy was **84.52%**, and the losses were **0.4477** and **0.4664**. These similar values indicate controlled generalization, but its **0.6912 macro-F1** is below the shallow MLP's score. Greater dense depth therefore did not produce a better gender representation.

**Class-level behaviour.** Men remained the strongest class at **0.895 F1**, followed by Women at **0.830** and Boys at **0.734**. Girls achieved **0.581 F1**. Unisex recall fell to **0.292**, even though its precision was **0.723**, resulting in the lowest class F1 of **0.416**. The model is therefore accurate when it commits to Unisex, but it assigns most true Unisex images to more common labels.

**Interpretation.** The small train–selection gap shows that overfitting is not the main issue. Instead, flattening the image and optimizing a deeper dense network still favours dominant categories and loses useful spatial structure. Since it trails the shallower model in both accuracy and macro-F1, the added depth is not justified for the selected gender classifier.


#### 5.2.5. Occasion: Model Architecture & Training Configuration

Configuration `deeper_mlp`; hidden dense units [256, 128, 64]. Adam 0.001, maximum 30 epochs, early-stopping patience 7.


In [ ]:
# Build the deeper MLP from the selected target-specific configuration.
config = SELECTED_CONFIGS['usage']['deeper_mlp']
method = config['method']
keras.utils.set_random_seed(SEED)

# Start with a fixed image input and training-only horizontal augmentation.
layers = [
    keras.Input(shape=(IMAGE_SIZE[1], IMAGE_SIZE[0], 3)),
    keras.layers.RandomFlip('horizontal'),
]
layers.append(keras.layers.Flatten())

# Add the configured hidden classifier layers and dropout regularization.
for width in config['widths']:
    layers.extend([keras.layers.Dense(width, activation='relu'), keras.layers.Dropout(0.2)])
layers.extend([keras.layers.Dense(len(labels_by_target['usage'])), ModelMetadata(name='metadata')])
models['usage'][method] = keras.Sequential(layers, name=method)
models['usage'][method].summary()

#### 5.2.6. Occasion: Compile & Train Model

The selection view controls epoch selection. Test images are excluded from training.


In [ ]:
# Compile and train the model while restoring the epoch with the best selection macro-F1.
model = models['usage'][method]

# Configure optimization and report both accuracy and imbalance-aware macro-F1.
model.compile(
    optimizer=keras.optimizers.Adam(0.001),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[
        keras.metrics.SparseCategoricalAccuracy(name='accuracy'),
        SupportedMacroF1(len(labels_by_target['usage'])),
    ],
)
loaders['usage']['train'].reset()

# Stop when selection macro-F1 no longer improves and restore its best weights.
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_macro_f1', mode='max', patience=7, restore_best_weights=True
    )
]

# Keep the selection partition read-only: it chooses epochs but never updates weights.
fitted = model.fit(
    loaders['usage']['train'],
    validation_data=loaders['usage']['selection'],
    epochs=30,
    callbacks=callbacks,
    shuffle=False,
    verbose=2,
)
histories['usage'][method] = pd.DataFrame(fitted.history)

#### 5.2.7. Occasion: Evaluation and Performance Metrics

Calculate validation accuracy and supported-class macro-F1, display the per-class classification report, and plot training and validation loss, accuracy and macro-F1. These measurements use the restored best epoch.


In [ ]:
# Evaluate the restored best weights, show class metrics, and plot the learning curves.
loader = loaders['usage']['selection']

# Convert logits to probabilities and preserve the loader's row order.
probabilities = np.vstack(
    [
        tf.nn.softmax(models['usage'][method](loader[i][0], training=False)).numpy()
        for i in range(len(loader))
    ]
)
predictions = probabilities.argmax(1)
display(
    pd.Series(
        dict(
            selection_accuracy=accuracy_score(loader.targets, predictions),
            selection_macro_f1=f1_score(
                loader.targets,
                predictions,
                labels=np.unique(loader.targets),
                average='macro',
                zero_division=0,
            ),
        )
    )
)

# Inspect every label because aggregate accuracy can hide minority-class failures.
per_class = pd.DataFrame(
    classification_report(
        loader.targets,
        predictions,
        labels=np.arange(len(labels_by_target['usage'])),
        target_names=labels_by_target['usage'],
        output_dict=True,
        zero_division=0,
    )
).T
display(per_class)
per_class.to_csv(RESULTS / f"{stem_for('usage')}_{method}_selection_per_class.csv")

# Plot train and selection curves to diagnose convergence and overfitting.
history = histories['usage'][method]
history.to_csv(RESULTS / f"{stem_for('usage')}_{method}_history.csv", index=False)
fig, axes = plt.subplots(1, 3, figsize=(14, 3))
for ax, metric in zip(axes, ['loss', 'accuracy', 'macro_f1']):
    ax.plot(np.arange(1, len(history) + 1), history[metric], label='Train')
    ax.plot(np.arange(1, len(history) + 1), history['val_' + metric], label='Selection')
    ax.set(xlabel='Epoch', ylabel=metric, title=method)
    ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / f"{stem_for('usage')}_{method}_learning_curves.png", dpi=160)
plt.show()

# Store one comparable row for the final cross-family decision.
selection_scores.setdefault('usage', {})[method] = dict(
    family='deeper_mlp',
    validation_accuracy=accuracy_score(loader.targets, predictions),
    validation_macro_f1=f1_score(
        loader.targets,
        predictions,
        labels=np.unique(loader.targets),
        average='macro',
        zero_division=0,
    ),
    complexity_parameters=models['usage'][method].count_params(),
    epochs_run=len(history),
)

#### 5.2.8. Occasion: Evaluation Analysis

**Learning behaviour.** The deeper MLP's best occasion macro-F1 occurred at epoch 23 of 30. Training and selection accuracy were nearly identical at **86.42%** and **86.55%**, and the losses were **0.4156** and **0.4069**. The absence of a generalization gap suggests that the model is not overfitting. Nevertheless, selection macro-F1 fell to **0.3694**, the lowest of the three occasion candidates, so stable learning did not translate into balanced class recognition.

**Class-level behaviour.** Casual achieved **0.917 F1** and Ethnic **0.788 F1**, but the model recovered only **43.0%** of Sports and **59.6%** of Formal examples. Their F1 scores were **0.539** and **0.711**. It produced zero recall for the NA/missing-usage, Party, Smart Casual and Travel classes. The high overall accuracy is therefore explained mainly by strong performance on Casual, which represents more than three quarters of the selection images.

**Interpretation.** Extra dense layers increased model size without improving the imbalanced objective. The network became more concentrated on common labels, as shown by its lower macro-F1 and four unrecovered classes. This result illustrates why architecture selection for occasion must prioritize per-class recall and macro-F1 rather than accuracy alone.


### 5.3. Convolutional Neural Network (CNN)

The selected CNN uses four 3 ? 3 convolution blocks (32, 64, 128, 256 filters), batch normalization, ReLU and max pooling, followed by a 2 ? 2 pooled grid and Dense(256). Only the selected continuation, when required, is fitted.


#### 5.3.1. Gender: Model Architecture & Training Configuration

Configuration `cnn_four_blocks_scheduled`; hidden dense units [256]. Initial Adam 0.001, maximum 30 epochs, early-stopping patience 7 and learning-rate halving after two unimproved epochs. Continuation, if specified, uses eight additional epochs with patience 4 and a fresh optimizer.


In [ ]:
# Build the CNN from the selected target-specific configuration.
config = SELECTED_CONFIGS['gender']['cnn']
method = config['method']
keras.utils.set_random_seed(SEED)

# Start with a fixed image input and training-only horizontal augmentation.
layers = [
    keras.Input(shape=(IMAGE_SIZE[1], IMAGE_SIZE[0], 3)),
    keras.layers.RandomFlip('horizontal'),
]

# Learn increasingly abstract local features while reducing spatial resolution.
for width in [32, 64, 128, 256]:
    layers.extend(
        [
            keras.layers.Conv2D(width, 3, padding='same'),
            keras.layers.BatchNormalization(),
            keras.layers.Activation('relu'),
            keras.layers.MaxPooling2D(2),
        ]
    )
h, w = IMAGE_SIZE[1] // 16, IMAGE_SIZE[0] // 16
layers.extend([keras.layers.AveragePooling2D((h // 2, w // 2)), keras.layers.Flatten()])

# Add the configured hidden classifier layers and dropout regularization.
for width in config['widths']:
    layers.extend([keras.layers.Dense(width, activation='relu'), keras.layers.Dropout(0.2)])
layers.extend([keras.layers.Dense(len(labels_by_target['gender'])), ModelMetadata(name='metadata')])
models['gender'][method] = keras.Sequential(layers, name=method)
models['gender'][method].summary()

#### 5.3.2. Gender: Compile & Train Model

The selection view controls epoch selection. Test images are excluded from training.


In [ ]:
# Compile and train the model while restoring the epoch with the best selection macro-F1.
model = models['gender'][method]

# Configure optimization and report both accuracy and imbalance-aware macro-F1.
model.compile(
    optimizer=keras.optimizers.Adam(0.001),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[
        keras.metrics.SparseCategoricalAccuracy(name='accuracy'),
        SupportedMacroF1(len(labels_by_target['gender'])),
    ],
)
loaders['gender']['train'].reset()

# Stop when selection macro-F1 no longer improves and restore its best weights.
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_macro_f1', mode='max', patience=7, restore_best_weights=True
    )
]
callbacks.insert(
    0, keras.callbacks.ReduceLROnPlateau(monitor='val_macro_f1', mode='max', factor=0.5, patience=2)
)

# Keep the selection partition read-only: it chooses epochs but never updates weights.
fitted = model.fit(
    loaders['gender']['train'],
    validation_data=loaders['gender']['selection'],
    epochs=30,
    callbacks=callbacks,
    shuffle=False,
    verbose=2,
)
histories['gender'][method] = pd.DataFrame(fitted.history)

#### 5.3.3. Gender: Evaluation and Performance Metrics

Calculate validation accuracy and supported-class macro-F1, display the per-class classification report, and plot training and validation loss, accuracy and macro-F1. These measurements use the restored best epoch.


In [ ]:
# Evaluate the restored best weights, show class metrics, and plot the learning curves.
loader = loaders['gender']['selection']

# Convert logits to probabilities and preserve the loader's row order.
probabilities = np.vstack(
    [
        tf.nn.softmax(models['gender'][method](loader[i][0], training=False)).numpy()
        for i in range(len(loader))
    ]
)
predictions = probabilities.argmax(1)
display(
    pd.Series(
        dict(
            selection_accuracy=accuracy_score(loader.targets, predictions),
            selection_macro_f1=f1_score(
                loader.targets,
                predictions,
                labels=np.unique(loader.targets),
                average='macro',
                zero_division=0,
            ),
        )
    )
)

# Inspect every label because aggregate accuracy can hide minority-class failures.
per_class = pd.DataFrame(
    classification_report(
        loader.targets,
        predictions,
        labels=np.arange(len(labels_by_target['gender'])),
        target_names=labels_by_target['gender'],
        output_dict=True,
        zero_division=0,
    )
).T
display(per_class)
per_class.to_csv(RESULTS / f"{stem_for('gender')}_{method}_selection_per_class.csv")

# Plot train and selection curves to diagnose convergence and overfitting.
history = histories['gender'][method]
history.to_csv(RESULTS / f"{stem_for('gender')}_{method}_history.csv", index=False)
fig, axes = plt.subplots(1, 3, figsize=(14, 3))
for ax, metric in zip(axes, ['loss', 'accuracy', 'macro_f1']):
    ax.plot(np.arange(1, len(history) + 1), history[metric], label='Train')
    ax.plot(np.arange(1, len(history) + 1), history['val_' + metric], label='Selection')
    ax.set(xlabel='Epoch', ylabel=metric, title=method)
    ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / f"{stem_for('gender')}_{method}_learning_curves.png", dpi=160)
plt.show()

# Store one comparable row for the final cross-family decision.
selection_scores.setdefault('gender', {})[method] = dict(
    family='cnn',
    validation_accuracy=accuracy_score(loader.targets, predictions),
    validation_macro_f1=f1_score(
        loader.targets,
        predictions,
        labels=np.unique(loader.targets),
        average='macro',
        zero_division=0,
    ),
    complexity_parameters=models['gender'][method].count_params(),
    epochs_run=len(history),
)

#### 5.3.4. Gender: Evaluation Analysis

**Learning behaviour.** The CNN reached its best gender macro-F1 at epoch 29 of 30. Training accuracy was **96.81%** and selection accuracy was **89.86%**, with losses of **0.1004** and **0.3662**. This is a visible train–selection gap and indicates some overfitting near the end of training. Retaining the best epoch limits its effect, and the resulting **0.7909 macro-F1** remains clearly above both MLPs.

**Class-level behaviour.** Men and Women achieved F1 scores of **0.932** and **0.904**. The smaller Boys and Girls classes also improved to **0.790** and **0.735 F1**, respectively. Unisex remained the hardest class, but recall rose to **0.522** and F1 to **0.594**, compared with recall of 0.360 for the shallow MLP and 0.292 for the deeper MLP. The CNN therefore improved both dominant-class accuracy and minority-class coverage.

**Interpretation.** Gender cues depend on garment shape and local design details, which the convolutional layers preserve more effectively than flattened-pixel MLPs. Although regularization or additional data could reduce the remaining generalization gap, the simultaneous gains in accuracy, macro-F1 and every minority-class F1 make the CNN the strongest gender candidate.


#### 5.3.5. Occasion: Model Architecture & Training Configuration

Configuration `cnn_continue_weighted`; hidden dense units [256]. Initial Adam 0.001, maximum 30 epochs, early-stopping patience 7 and learning-rate halving after two unimproved epochs. Continuation, if specified, uses eight additional epochs with patience 4 and a fresh optimizer.


In [ ]:
# Build the CNN from the selected target-specific configuration.
config = SELECTED_CONFIGS['usage']['cnn']
method = config['method']
keras.utils.set_random_seed(SEED)

# Start with a fixed image input and training-only horizontal augmentation.
layers = [
    keras.Input(shape=(IMAGE_SIZE[1], IMAGE_SIZE[0], 3)),
    keras.layers.RandomFlip('horizontal'),
]

# Learn increasingly abstract local features while reducing spatial resolution.
for width in [32, 64, 128, 256]:
    layers.extend(
        [
            keras.layers.Conv2D(width, 3, padding='same'),
            keras.layers.BatchNormalization(),
            keras.layers.Activation('relu'),
            keras.layers.MaxPooling2D(2),
        ]
    )
h, w = IMAGE_SIZE[1] // 16, IMAGE_SIZE[0] // 16
layers.extend([keras.layers.AveragePooling2D((h // 2, w // 2)), keras.layers.Flatten()])

# Add the configured hidden classifier layers and dropout regularization.
for width in config['widths']:
    layers.extend([keras.layers.Dense(width, activation='relu'), keras.layers.Dropout(0.2)])
layers.extend([keras.layers.Dense(len(labels_by_target['usage'])), ModelMetadata(name='metadata')])
models['usage'][method] = keras.Sequential(layers, name=method)
models['usage'][method].summary()

#### 5.3.6. Occasion: Compile & Train Model

The selection view controls epoch selection. Test images are excluded from training.


In [ ]:
# Compile and train the model while restoring the epoch with the best selection macro-F1.
model = models['usage'][method]

# Configure optimization and report both accuracy and imbalance-aware macro-F1.
model.compile(
    optimizer=keras.optimizers.Adam(0.001),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[
        keras.metrics.SparseCategoricalAccuracy(name='accuracy'),
        SupportedMacroF1(len(labels_by_target['usage'])),
    ],
)
loaders['usage']['train'].reset()

# Stop when selection macro-F1 no longer improves and restore its best weights.
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_macro_f1', mode='max', patience=7, restore_best_weights=True
    )
]
callbacks.insert(
    0, keras.callbacks.ReduceLROnPlateau(monitor='val_macro_f1', mode='max', factor=0.5, patience=2)
)

# Keep the selection partition read-only: it chooses epochs but never updates weights.
fitted = model.fit(
    loaders['usage']['train'],
    validation_data=loaders['usage']['selection'],
    epochs=30,
    callbacks=callbacks,
    shuffle=False,
    verbose=2,
)
histories['usage'][method] = pd.DataFrame(fitted.history)
histories['usage'][method].to_csv(RESULTS / 'usage_cnn_initial_history.csv', index=False)

# Begin the low-learning-rate continuation from the initial CNN weights.
# Reset the loader so continuation starts from a reproducible order.
keras.utils.set_random_seed(SEED)
loaders['usage']['train'].reset()
model.compile(
    optimizer=keras.optimizers.Adam(3e-05),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[
        keras.metrics.SparseCategoricalAccuracy(name='accuracy'),
        SupportedMacroF1(len(labels_by_target['usage'])),
    ],
)
class_weight = None

# Use capped square-root class weights to help rare labels without destabilizing training.
counts = np.bincount(loaders['usage']['train'].targets, minlength=len(labels_by_target['usage']))
weights = np.sqrt(counts.sum() / (len(counts) * counts))
weights = np.clip(weights / np.average(weights, weights=counts), 0.25, 6)
class_weight = dict(enumerate(weights))
fitted = model.fit(
    loaders['usage']['train'],
    validation_data=loaders['usage']['selection'],
    epochs=8,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor='val_macro_f1', mode='max', patience=4, restore_best_weights=True
        )
    ],
    class_weight=class_weight,
    shuffle=False,
    verbose=2,
)
histories['usage'][method] = pd.DataFrame(fitted.history)

#### 5.3.7. Occasion: Evaluation and Performance Metrics

Calculate validation accuracy and supported-class macro-F1, display the per-class classification report, and plot training and validation loss, accuracy and macro-F1. These measurements use the restored best epoch.


In [ ]:
# Evaluate the restored best weights, show class metrics, and plot the learning curves.
loader = loaders['usage']['selection']

# Convert logits to probabilities and preserve the loader's row order.
probabilities = np.vstack(
    [
        tf.nn.softmax(models['usage'][method](loader[i][0], training=False)).numpy()
        for i in range(len(loader))
    ]
)
predictions = probabilities.argmax(1)
display(
    pd.Series(
        dict(
            selection_accuracy=accuracy_score(loader.targets, predictions),
            selection_macro_f1=f1_score(
                loader.targets,
                predictions,
                labels=np.unique(loader.targets),
                average='macro',
                zero_division=0,
            ),
        )
    )
)

# Inspect every label because aggregate accuracy can hide minority-class failures.
per_class = pd.DataFrame(
    classification_report(
        loader.targets,
        predictions,
        labels=np.arange(len(labels_by_target['usage'])),
        target_names=labels_by_target['usage'],
        output_dict=True,
        zero_division=0,
    )
).T
display(per_class)
per_class.to_csv(RESULTS / f"{stem_for('usage')}_{method}_selection_per_class.csv")

# Plot train and selection curves to diagnose convergence and overfitting.
history = histories['usage'][method]
history.to_csv(RESULTS / f"{stem_for('usage')}_{method}_history.csv", index=False)
fig, axes = plt.subplots(1, 3, figsize=(14, 3))
for ax, metric in zip(axes, ['loss', 'accuracy', 'macro_f1']):
    ax.plot(np.arange(1, len(history) + 1), history[metric], label='Train')
    ax.plot(np.arange(1, len(history) + 1), history['val_' + metric], label='Selection')
    ax.set(xlabel='Epoch', ylabel=metric, title=method)
    ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / f"{stem_for('usage')}_{method}_learning_curves.png", dpi=160)
plt.show()

# Store one comparable row for the final cross-family decision.
selection_scores.setdefault('usage', {})[method] = dict(
    family='cnn',
    validation_accuracy=accuracy_score(loader.targets, predictions),
    validation_macro_f1=f1_score(
        loader.targets,
        predictions,
        labels=np.unique(loader.targets),
        average='macro',
        zero_division=0,
    ),
    complexity_parameters=models['usage'][method].count_params(),
    epochs_run=len(history),
)

#### 5.3.8. Occasion: Evaluation Analysis

**Learning behaviour.** During the recorded continuation stage, the weighted CNN reached its best occasion macro-F1 at epoch 6 of 8. Training accuracy was **95.26%** and selection accuracy was **89.55%**; training and selection losses were **0.1797** and **0.3138**. Because the training objective uses class weights while selection loss is unweighted, the two loss values are not directly equivalent. The accuracy gap still indicates some overfitting, but early stopping retains the strongest selection epoch.

**Class-level behaviour.** Casual achieved **0.934 F1**, while Ethnic and Formal improved to **0.867** and **0.811 F1**. Sports precision and recall were both **0.700**, a substantial recall gain over the two MLPs. The NA/missing-usage label recovered one of five examples. Party, Smart Casual and Travel remained unrecovered; their combined selection support was only seven images, so their individual estimates are also highly unstable. These failures keep macro-F1 at **0.4497** despite the model's **89.55% accuracy**.

**Interpretation.** The CNN gives the best overall accuracy and the strongest results on the four practically supported occasion classes. Its macro-F1 is slightly below the shallow MLP because macro averaging gives the three unrecovered tiny classes the same weight as Casual. The weighted CNN is therefore the most useful deployed candidate among those tested, while its rare-class results identify a data-coverage limitation that architecture changes alone cannot reliably resolve.


## 6. Ultimate Judgement

The three model families are compared using the validation metrics already calculated in their evaluation sections. The model with the highest validation macro-F1 is selected. Accuracy and parameter count are used to interpret the result and would break an exact macro-F1 tie.


In [ ]:
# Prepare containers for the cross-family comparison and final checkpoints.
comparisons, winners, checkpoints = {}, {}, {}

In [ ]:
# Rank the three model families by macro-F1, then accuracy, then parameter count.
comparison = pd.DataFrame(selection_scores['gender']).T
comparison = comparison.sort_values(
    ['validation_macro_f1', 'validation_accuracy', 'complexity_parameters'],
    ascending=[False, False, True],
)
winners['gender'] = comparison.index[0]
comparison['selected'] = comparison.index == winners['gender']
comparisons['gender'] = comparison
display(comparison)
print('Selected:', winners['gender'])
comparison.to_csv(RESULTS / 'gender_comparison.csv')

**Model comparison: gender**

| Family | Configuration | Validation accuracy | Macro-F1 | Parameters |
|---|---|---:|---:|---:|
| shallow_mlp | shallow_mlp_lower_lr | 84.69% | 0.7118 | 9,438,725 |
| deeper_mlp | deeper_mlp_two_layers | 84.52% | 0.6912 | 9,470,981 |
| cnn | cnn_four_blocks_scheduled | 89.86% | 0.7909 | 654,021 |


In [ ]:
# Rank the three model families by macro-F1, then accuracy, then parameter count.
comparison = pd.DataFrame(selection_scores['usage']).T
comparison = comparison.sort_values(
    ['validation_macro_f1', 'validation_accuracy', 'complexity_parameters'],
    ascending=[False, False, True],
)
winners['usage'] = comparison.index[0]
comparison['selected'] = comparison.index == winners['usage']
comparisons['usage'] = comparison
display(comparison)
print('Selected:', winners['usage'])
comparison.to_csv(RESULTS / 'usage_comparison.csv')

**Model comparison: usage**

| Family | Configuration | Validation accuracy | Macro-F1 | Parameters |
|---|---|---:|---:|---:|
| shallow_mlp | shallow_mlp_lower_lr | 87.38% | 0.4522 | 9,439,753 |
| deeper_mlp | deeper_mlp | 86.55% | 0.3694 | 9,479,177 |
| cnn | cnn_continue_weighted | 89.55% | 0.4497 | 655,049 |


In [ ]:
# Define imbalance-aware performance and probability-calibration metrics for the final test.
def supported_macro_f1(truth, predictions) -> float:
    """Macro-average over labels present in the ground-truth partition."""
    return float(
        f1_score(truth, predictions, labels=np.unique(truth), average="macro", zero_division=0)
    )


def expected_calibration_error(
    truth: np.ndarray,
    probabilities: np.ndarray,
    bins: int = 10,
) -> float:
    """Compute top-label expected calibration error."""
    truth = np.asarray(truth)
    probabilities = np.asarray(probabilities)
    confidence = probabilities.max(axis=1)
    correct = probabilities.argmax(axis=1) == truth
    edges = np.linspace(0.0, 1.0, bins + 1)
    total = len(truth)
    error = 0.0
    for lower, upper in zip(edges[:-1], edges[1:], strict=True):
        selected = (confidence > lower) & (confidence <= upper)
        if selected.any():
            error += (
                selected.sum()
                / total
                * abs(float(correct[selected].mean()) - float(confidence[selected].mean()))
            )
    return float(error)

### 6.1. Selected-Model Evaluation

Evaluate only the selected model on the internal test partition. This cell reports accuracy, supported-class macro-F1, calibration error, negative log-likelihood, Brier score and a per-class classification report. The internal test has prior development exposure and is not used to revise the selected architecture.


In [ ]:
# Evaluate each selected family winner once on the untouched internal-test partition.
test_results = {}
checkpoints = {}

method = winners['gender']
model = models['gender'][method]
test_frame = metadata_by_target['gender'].loc[metadata_by_target['gender']['split'].eq('test')]
test_loader = CachedBatches(
    test_frame, 'gender', labels_by_target['gender'], normalisation, batch_size=BATCH_SIZE
)

# Convert logits to probabilities and preserve the loader's row order.
probabilities = np.vstack(
    [
        tf.nn.softmax(model(test_loader[i][0], training=False)).numpy()
        for i in range(len(test_loader))
    ]
)
truth = test_loader.targets
predictions = probabilities.argmax(1)

# Measure label accuracy, class balance, and probability quality on the final test.
metrics = {
    'accuracy': float(accuracy_score(truth, predictions)),
    'macro_f1': supported_macro_f1(truth, predictions),
    'ece': expected_calibration_error(truth, probabilities),
    'nll': float(log_loss(truth, probabilities, labels=np.arange(len(labels_by_target['gender'])))),
    'brier': float(
        np.mean(
            np.sum((probabilities - np.eye(len(labels_by_target['gender']))[truth]) ** 2, axis=1)
        )
    ),
}
test_results['gender'] = dict(
    labels=labels_by_target['gender'],
    truth=truth,
    predictions=predictions,
    probabilities=probabilities,
    metrics=metrics,
)

# Bundle the fitted model with everything required to reproduce inference.
checkpoints['gender'] = dict(
    target='gender',
    labels=labels_by_target['gender'],
    model=model,
    model_type=method,
    temperature=1.0,
    review_policy=None,
    mean=normalisation['mean'],
    std=normalisation['std'],
    image_size=list(IMAGE_SIZE),
    comparison_row=comparisons['gender'].loc[method].to_dict(),
    test_metrics=metrics,
)
display(pd.Series(metrics, name='gender'))

# Inspect every label because aggregate accuracy can hide minority-class failures.
per_class = pd.DataFrame(
    classification_report(
        truth,
        predictions,
        labels=np.arange(len(labels_by_target['gender'])),
        target_names=labels_by_target['gender'],
        output_dict=True,
        zero_division=0,
    )
).T
display(per_class)
per_class.to_csv(RESULTS / 'gender_test_per_class.csv')
pd.Series(metrics).to_csv(RESULTS / 'gender_test_metrics.csv')

method = winners['usage']
model = models['usage'][method]
test_frame = metadata_by_target['usage'].loc[metadata_by_target['usage']['split'].eq('test')]
test_loader = CachedBatches(
    test_frame, 'usage', labels_by_target['usage'], normalisation, batch_size=BATCH_SIZE
)
probabilities = np.vstack(
    [
        tf.nn.softmax(model(test_loader[i][0], training=False)).numpy()
        for i in range(len(test_loader))
    ]
)
truth = test_loader.targets
predictions = probabilities.argmax(1)
metrics = {
    'accuracy': float(accuracy_score(truth, predictions)),
    'macro_f1': supported_macro_f1(truth, predictions),
    'ece': expected_calibration_error(truth, probabilities),
    'nll': float(log_loss(truth, probabilities, labels=np.arange(len(labels_by_target['usage'])))),
    'brier': float(
        np.mean(
            np.sum((probabilities - np.eye(len(labels_by_target['usage']))[truth]) ** 2, axis=1)
        )
    ),
}
test_results['usage'] = dict(
    labels=labels_by_target['usage'],
    truth=truth,
    predictions=predictions,
    probabilities=probabilities,
    metrics=metrics,
)
checkpoints['usage'] = dict(
    target='usage',
    labels=labels_by_target['usage'],
    model=model,
    model_type=method,
    temperature=1.0,
    review_policy=None,
    mean=normalisation['mean'],
    std=normalisation['std'],
    image_size=list(IMAGE_SIZE),
    comparison_row=comparisons['usage'].loc[method].to_dict(),
    test_metrics=metrics,
)
display(pd.Series(metrics, name='usage'))
per_class = pd.DataFrame(
    classification_report(
        truth,
        predictions,
        labels=np.arange(len(labels_by_target['usage'])),
        target_names=labels_by_target['usage'],
        output_dict=True,
        zero_division=0,
    )
).T
display(per_class)
per_class.to_csv(RESULTS / 'usage_test_per_class.csv')
pd.Series(metrics).to_csv(RESULTS / 'usage_test_metrics.csv')

### 6.2. Selected-Model Performance

**Recorded selected-model performance.**

| Target | Selected model | Validation accuracy | Validation macro-F1 | Internal-test accuracy | Internal-test macro-F1 |
|---|---|---:|---:|---:|---:|
| gender | cnn_four_blocks_scheduled | 89.86% | 0.7909 | 88.61% | 0.7396 |
| usage | shallow_mlp_lower_lr | 87.38% | 0.4522 | 85.69% | 0.3745 |

Models are selected by validation macro-F1. Internal-test scores have prior development exposure and are descriptive, not an independent model-selection criterion.


In [ ]:
# Display the concise final judgement for the chosen model and its internal-test result.
winner = comparisons['gender'].loc[winners['gender']]
baseline = comparisons['gender'].loc[comparisons['gender']['family'].eq('shallow_mlp')].iloc[0]
display(
    pd.Series(
        {
            'selected_method': winners['gender'],
            'selection_macro_f1': winner.validation_macro_f1,
            'selection_accuracy': winner.validation_accuracy,
            'macro_f1_gain_over_baseline': winner.validation_macro_f1
            - baseline.validation_macro_f1,
            'internal_test_accuracy': test_results['gender']['metrics']['accuracy'],
            'internal_test_macro_f1': test_results['gender']['metrics']['macro_f1'],
        },
        name='gender',
    )
)
winner = comparisons['usage'].loc[winners['usage']]
baseline = comparisons['usage'].loc[comparisons['usage']['family'].eq('shallow_mlp')].iloc[0]
display(
    pd.Series(
        {
            'selected_method': winners['usage'],
            'selection_macro_f1': winner.validation_macro_f1,
            'selection_accuracy': winner.validation_accuracy,
            'macro_f1_gain_over_baseline': winner.validation_macro_f1
            - baseline.validation_macro_f1,
            'internal_test_accuracy': test_results['usage']['metrics']['accuracy'],
            'internal_test_macro_f1': test_results['usage']['metrics']['macro_f1'],
        },
        name='usage',
    )
)

### 6.3. Decision Analysis

For gender, the 256-unit shallow MLP reaches 0.7118 validation macro-F1; the two-layer deeper MLP (256, 128) reaches 0.6912. Two hidden layers improved the deeper-family result relative to the previously measured three-layer choice (0.6409). The four-block CNN remains strongest at 0.7909 macro-F1 and 89.86% validation accuracy, with 654,021 parameters. The architecture is unchanged; a small difference between training runs is not evidence of an architectural gain. Its internal-test accuracy is 88.61% and macro-F1 is 0.7396. Predictions describe catalogue audience labels, not personal identity.

For occasion, the 256-unit shallow MLP wins narrowly at 0.4522 validation macro-F1, against 0.4497 for weighted CNN continuation and 0.3694 for the deeper MLP. The deeper family retains three hidden layers (256, 128, 64) with Adam 0.001 because that measured configuration exceeded the tested alternatives. This differs from gender because configuration selection follows each target's validation evidence, not a requirement to use different architectures.

The occasion CNN has higher validation accuracy (89.55% versus 87.38%) and fewer parameters, while the MLP wins the declared macro-F1 criterion. Its small advantage depends on very limited rare-class support and should be treated as provisional. The selected MLP records 85.69% internal-test accuracy and 0.3745 macro-F1. Neither a high overall accuracy nor a small single-seed F1 lead establishes broad minority-class reliability. Confidence scores do not remove annotation ambiguity or the internal test's prior development exposure.


## 7. Final Prediction

Save the selected Keras model with its label order and preprocessing metadata, reload it, and run one sample prediction. This functional check uses softmax probabilities with temperature 1.0; it does not add a separate confidence-calibration or review policy.


In [ ]:
# Export a self-contained inference model with preprocessing and label metadata embedded.
def save_checkpoint(checkpoint, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path = path.with_suffix(".keras")
    model = checkpoint["model"]

    # Separate serializable metadata from the live Keras model object.
    metadata = {key: value for key, value in checkpoint.items() if key != "model"}
    metadata = json.loads(json.dumps(metadata, default=lambda value: value.item()))
    model.get_layer("metadata").metadata = metadata

    # Export architecture, learned weights and metadata without optimizer slots.
    # Rebuild without optimizer state to keep the deployment file smaller.
    inference_model = type(model).from_config(model.get_config())
    inference_model.set_weights(model.get_weights())
    inference_model.save(path)
    return path

In [ ]:
# Save the selected model, its learning history, and a machine-readable experiment summary.
stem, method = (stem_for('gender'), winners['gender'])
path = save_checkpoint(  # Bundle the fitted model with everything required to reproduce inference.
    checkpoints['gender'], OUTPUT / f'{stem}_model.keras'
)
histories['gender'][method].to_csv(RESULTS / f'{stem}_history.csv', index=False)
summary = dict(
    target='gender',
    selected=method,
    cnn_selected=SELECTED_CONFIGS['gender']['cnn']['method'],
    framework='tensorflow_keras',
    model_file=path.name,
    test_metrics=test_results['gender']['metrics'],
    split_sizes={name: len(frame) for name, frame in frames['gender'].items()},
    test_scope='internal_test_with_prior_development_exposure',
)
(RESULTS / f'{stem}_summary.json').write_text(json.dumps(summary, indent=2) + '\n')
print('Saved', path)
stem, method = (stem_for('usage'), winners['usage'])
path = save_checkpoint(checkpoints['usage'], OUTPUT / f'{stem}_model.keras')
histories['usage'][method].to_csv(RESULTS / f'{stem}_history.csv', index=False)
summary = dict(
    target='usage',
    selected=method,
    cnn_selected=SELECTED_CONFIGS['usage']['cnn']['method'],
    framework='tensorflow_keras',
    model_file=path.name,
    test_metrics=test_results['usage']['metrics'],
    split_sizes={name: len(frame) for name, frame in frames['usage'].items()},
    test_scope='internal_test_with_prior_development_exposure',
)
(RESULTS / f'{stem}_summary.json').write_text(json.dumps(summary, indent=2) + '\n')
print('Saved', path)

### 7.1. Load the Saved Model & Predict

A successful reload checks the saved architecture and metadata. This example is a functional check, not an independent quality estimate.


In [ ]:
# Reload an exported Keras classifier and recover the metadata stored inside it.
def resolve_classifier_path(path):
    path = Path(path)
    if path.exists():
        return path
    raise FileNotFoundError(
        f"Missing trained classifier: {path}. Run the classification notebook first."
    )

def load_checkpoint(path):
    path = resolve_classifier_path(path)
    if path.suffix == ".keras":
        model = keras.models.load_model(path, compile=False)
        return {**dict(model.get_layer("metadata").metadata), "model": model}
    raise ValueError(f"Unsupported classifier format: {path.suffix}")

In [ ]:
# Run a single-image smoke test through the same preprocessing path used by the application.
checkpoint = load_checkpoint(OUTPUT / f"{stem_for('gender')}_model.keras")

# Apply the embedded preprocessing values rather than notebook globals.
with Image.open(frames['gender']['selection'].image_path.iloc[0]) as image:
    resized = image.convert('RGB').resize(
        tuple(checkpoint['image_size']), Image.Resampling.BILINEAR
    )
    inputs = (
        np.asarray(resized, dtype=np.float32) / 255.0 - np.asarray(checkpoint['mean'])
    ) / np.asarray(checkpoint['std'])
probabilities = tf.nn.softmax(
    checkpoint['model'](inputs[None, ...], training=False), axis=-1
).numpy()[0]
indices = np.argsort(probabilities)[::-1][:3]
print(
    {
        'target': 'gender',
        'label': checkpoint['labels'][int(indices[0])],
        'confidence': float(probabilities[indices[0]]),
    }
)
checkpoint = load_checkpoint(OUTPUT / f"{stem_for('usage')}_model.keras")
with Image.open(frames['usage']['selection'].image_path.iloc[0]) as image:
    resized = image.convert('RGB').resize(
        tuple(checkpoint['image_size']), Image.Resampling.BILINEAR
    )
    inputs = (
        np.asarray(resized, dtype=np.float32) / 255.0 - np.asarray(checkpoint['mean'])
    ) / np.asarray(checkpoint['std'])
probabilities = tf.nn.softmax(
    checkpoint['model'](inputs[None, ...], training=False), axis=-1
).numpy()[0]
indices = np.argsort(probabilities)[::-1][:3]
print(
    {
        'target': 'usage',
        'label': checkpoint['labels'][int(indices[0])],
        'confidence': float(probabilities[indices[0]]),
    }
)

## 8. Conclusion

This task developed two related catalogue classifiers: gender and occasion. Both targets were trained from the same audited fashion images and frozen group-aware partitions, but they were evaluated independently because their class distributions and visual cues differ. A shallow MLP, a deeper MLP and a CNN were compared for each target using selection macro-F1 as the primary criterion, followed by accuracy and parameter count only when needed. This target-specific selection process avoids assuming that one architecture must be optimal for every label.

For gender, the shallow MLP achieved **84.69% selection accuracy** and **0.7118 macro-F1**, while the deeper MLP reached **84.52% accuracy** and **0.6912 macro-F1**. The four-block CNN was clearly stronger, achieving **89.86% accuracy** and **0.7909 macro-F1** with 654,021 parameters instead of approximately 9.5 million. It improved F1 for Men and Women to **0.932** and **0.904**, while the smaller Boys and Girls classes reached **0.790** and **0.735**. Unisex remained the most difficult label, but its recall increased to **0.522**, compared with 0.360 for the shallow MLP and 0.292 for the deeper MLP. These results show that convolutional features improve both dominant-class accuracy and minority-class coverage.

The selected gender CNN achieved **88.61% internal-test accuracy** and **0.7396 macro-F1**. Its training accuracy was higher than selection accuracy, so some overfitting remains, but early stopping preserves the best observed epoch. The internal-test result supports the CNN choice within the supplied data while also showing that performance is less balanced than accuracy alone suggests. These predictions represent audience categories assigned in the catalogue; they should not be interpreted as determining a person's identity from an image.

Occasion produced a different result. The shallow MLP achieved the highest selection macro-F1 at **0.4522**, narrowly exceeding the weighted CNN's **0.4497**, while the deeper MLP reached **0.3694**. Under the declared selection rule, the shallow MLP is therefore retained. The CNN nevertheless achieved the highest selection accuracy (**89.55%** versus **87.38%**) and improved the supported Casual, Ethnic, Formal and Sports classes. The shallow MLP's macro-F1 advantage came partly from recovering two of the five NA/missing-usage examples, while both models failed to recover Party, Smart Casual and Travel, whose combined selection support was only seven images. The narrow difference should consequently be treated as provisional rather than evidence that dense networks are generally better for occasion prediction.

On the internal-test partition, the selected occasion MLP achieved **85.69% accuracy** and **0.3745 macro-F1**. This large gap confirms that strong recognition of the dominant Casual class does not guarantee dependable minority-class performance. The result is constrained primarily by extreme class imbalance and very small support for several labels. More layers cannot reliably learn categories that have only a handful of examples, and confidence values cannot resolve inconsistent or ambiguous catalogue annotations.

Both selected models are exported in Keras format with their labels and preprocessing settings embedded for consistent use by the web application and prediction API. The gender CNN is the clear architectural winner, whereas the occasion result demonstrates why each target requires its own evidence-based decision. Future work should prioritize additional examples for Unisex and rare occasions, review whether extremely sparse occasion labels should be merged or recollected, evaluate multiple training seeds, and test the final models on an independent catalogue source.
